# Dự báo PM2.5 giờ tiếp theo tại TP.HCM — bản tái lập trên Google Colab

Notebook này chạy **trực tiếp pipeline của repository**, không sao chép logic huấn luyện. Notebook và CLI dùng cùng mã nguồn, cấu hình, dữ liệu và `random_state`. Lệnh CLI tương ứng là `python -m src.pipeline train --config configs/config.yaml --no-artifacts`.

> Phạm vi: dữ liệu mẫu trong repository dùng để kiểm tra khả năng tái lập kỹ thuật. Kết quả không phải cảnh báo sức khỏe chính thức.

## 1. Chuẩn bị repository và môi trường

In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/haminhthong/hcmc-pm25-forecasting.git"
REPOSITORY_NAME = "hcmc-pm25-forecasting"
IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    project_root = Path("/content") / REPOSITORY_NAME
    if not project_root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPOSITORY_URL, str(project_root)],
            check=True,
        )
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    project_root = next(
        (path for path in candidates if (path / "pyproject.toml").exists()),
        None,
    )
    if project_root is None:
        raise FileNotFoundError("Không tìm thấy thư mục gốc của dự án.")

os.chdir(project_root)
print(f"Thư mục dự án: {project_root}")
print(f"Môi trường: {'Google Colab' if IN_COLAB else 'máy cục bộ'}")

In [ ]:
# Colab cần đúng phiên bản thư viện đã dùng để tạo kết quả chuẩn.
if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"],
        check=True,
    )
    print("Đã cài đặt môi trường tái lập cho Colab.")
else:
    print("Bỏ qua cài đặt vì đang chạy trong môi trường phát triển cục bộ.")

## 2. Kiểm tra dữ liệu đầu vào

Pipeline tự tìm dữ liệu theo `configs/config.yaml`, kiểm tra schema, sắp xếp theo trạm–thời gian và tạo đặc trưng theo thời điểm thực để tránh lấy dữ liệu tương lai.

In [ ]:
import json
import platform

import numpy as np
import pandas as pd
import sklearn
import yaml

from src.data import audit_air_quality, load_air_quality, load_config

config = load_config("configs/config.yaml")
data = load_air_quality(config)
audit = audit_air_quality(data, config)

environment = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "pyyaml": yaml.__version__,
}
print("Phiên bản môi trường:", json.dumps(environment, ensure_ascii=False, indent=2))
pd.DataFrame({
    "thuộc tính": ["Số dòng", "Số trạm", "Thời gian bắt đầu", "Thời gian kết thúc", "Dòng trùng", "Khoảng giờ không đều"],
    "giá trị": [audit["rows"], audit["stations"], audit["period"][0], audit["period"][1], audit["duplicate_station_timestamps"], audit["irregular_hourly_gaps"]],
})

## 3. Huấn luyện và đánh giá

Hàm dưới đây chính là hàm được CLI gọi. `persist_artifacts=False` giúp notebook không ghi đè mô hình đang dùng cho demo.

In [ ]:
from src.pipeline import run_train_pipeline

result = run_train_pipeline("configs/config.yaml", persist_artifacts=False)
print(f"Mô hình tốt nhất trên CV: {result['best_cv_model']}")
print(f"Cách dự báo được chọn: {result['forecast_strategy']}")

In [ ]:
metrics = result["metrics"]
baseline = result["evaluation"]["baselines"]["persistence"]
summary = pd.DataFrame(
    [
        {"mô hình": result["forecast_strategy"], "MAE": metrics["mae"], "RMSE": metrics["rmse"], "Macro-F1": metrics["macro_f1"], "QWK": metrics["qwk"]},
        {"mô hình": "persistence", "MAE": baseline["mae"], "RMSE": baseline["rmse"], "Macro-F1": baseline["macro_f1"], "QWK": baseline["qwk"]},
    ]
).set_index("mô hình")
summary.round(4)

In [ ]:
backtest_rows = []
for model_name, values in result["evaluation"]["backtest"].items():
    backtest_rows.append({
        "mô hình": model_name,
        "MAE trung bình": values["mae_mean"],
        "Độ lệch chuẩn MAE": values["mae_std"],
        "RMSE trung bình": values["rmse_mean"],
    })
pd.DataFrame(backtest_rows).set_index("mô hình").round(4)

## 4. Xác minh kết quả tái lập

Cell này kiểm tra hash dữ liệu, tên mô hình và các chỉ số quan trọng với snapshot đã lưu trong repository. Sai khác số thực nhỏ hơn `1e-9` được xem là nhiễu tính toán, không phải khác biệt mô hình.

In [ ]:
with open("configs/reproducibility_expected.json", encoding="utf-8") as file:
    expected = json.load(file)

assert result["metadata"]["data_provenance"]["data_sha256"] == expected["data_sha256"], "Dữ liệu đầu vào không còn giống snapshot chuẩn."
assert result["best_cv_model"] == expected["model_name"], "Mô hình được chọn đã thay đổi."

for metric_name, expected_value in expected["metrics"].items():
    actual_value = result["metrics"][metric_name]
    np.testing.assert_allclose(actual_value, expected_value, rtol=0, atol=1e-9)

np.testing.assert_allclose(
    result["evaluation"]["backtest"]["ridge"]["mae_mean"],
    expected["ridge_backtest_mae_mean"],
    rtol=0,
    atol=1e-9,
)
print("✅ Kết quả khớp snapshot chuẩn: dữ liệu, mô hình và các metric đều tái lập thành công.")

## 5. Kết luận

- Notebook và CLI dùng chung `src.pipeline`, nên không phát sinh hai phiên bản logic.
- Kết quả hiện tại **không vượt persistence baseline** trên dữ liệu mẫu; đây là kết quả trung thực và chiến lược dự báo được chọn là `persistence`.
- Muốn huấn luyện và lưu bộ file kết quả, chạy `python -m src.pipeline train --config configs/config.yaml` trong terminal.